# NB5 — Cost accounting: what each component actually costs

The CIKM version reported 275.6 ms average latency on a "persistent 100K-passage
index" and claimed the retrieval and filtering stack is practical. Two problems
with that number as evidence:

1. **The measured corpus was synthetic.** The latency script generates documents
   by sampling from a 12-word vocabulary. BM25 over a 12-word vocabulary has
   nothing like the posting-list structure of a real corpus, and the encoder sees
   trivially short, repetitive inputs.
2. **The claim it supports is about the whole pipeline**, but the numbers were
   collected under a configuration that differs from the one used for the quality
   results (a different candidate count, a different reranking depth). Two of the
   repository's own artifacts disagree with the paper by an order of magnitude
   (`latency_results.txt` records a 3,325 ms median).

This notebook measures cost on **real corpora, in the exact configuration the
quality results were produced under**, and reports it as a cost/quality curve
rather than a single number. Both halves of the tradeoff come from the same runs.

Reported per configuration:

- wall-clock p50/p95/p99 per query, split by stage (encode / dense / BM25 / fuse /
  rerank / filter);
- index build time and resident memory;
- **cross-encoder forward passes per query**, which is the hardware-independent
  cost measure and the one that actually transfers to another machine;
- the quality obtained at that cost, so the table is a frontier and not a boast.

**Runtime** ~20 min on a free T4. Peak VRAM ~3 GB.

In [ ]:
!pip install -q "sentence-transformers>=3.0" "bm25s[full]" PyStemmer datasets pyarrow psutil 2>&1 | tail -1

In [ ]:
import os, sys, json, math, time, random, hashlib, re, gc, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# torch is required by the experiment notebooks but not by the reporting one,
# so a missing install degrades to a clear message rather than a traceback.
try:
    import torch
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cuda":
        props = torch.cuda.get_device_properties(0)
        print(f"GPU: {props.name} | {props.total_memory/1e9:.1f} GB | "
              f"n_gpus={torch.cuda.device_count()}")
    else:
        print("WARNING: no GPU detected. Everything will run but ~10-20x slower.")
except ImportError:
    torch = None
    DEVICE = "cpu"
    print("torch not installed (fine for the reporting notebook, required for the rest).")

# Artifact directory. On Kaggle write to /kaggle/working so results persist in
# the version output; on Colab mount Drive if you want results to survive a
# disconnect.
if Path("/kaggle/working").exists():
    ART = Path("/kaggle/working/cognisync_tmlr")
elif Path("/content/drive/MyDrive").exists():
    ART = Path("/content/drive/MyDrive/cognisync_tmlr")
else:
    ART = Path("./cognisync_tmlr")
(ART / "results").mkdir(parents=True, exist_ok=True)
(ART / "cache").mkdir(parents=True, exist_ok=True)
print("Artifacts ->", ART)


def save_json(obj, name):
    p = ART / "results" / name
    with open(p, "w") as f:
        json.dump(obj, f, indent=2, default=float)
    print("saved", p)
    return p


def save_csv(df, name):
    p = ART / "results" / name
    df.to_csv(p, index=False)
    print("saved", p, df.shape)
    return p

import psutil

BENCH_DATASETS = ["scifact", "fiqa"]     # real corpora, 5K and 58K
N_BENCH_QUERIES = 200
DEPTHS = [100, 1000]                     # first-stage depth
RERANK_BUDGETS = [0, 10, 50, 100]
WARMUP = 10

CONFIG = dict(datasets=BENCH_DATASETS, n_queries=N_BENCH_QUERIES,
              depths=DEPTHS, rerank_budgets=RERANK_BUDGETS, seed=SEED,
              device=DEVICE, gpu=(torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"))
print(json.dumps(CONFIG, indent=2))


def rss_gb():
    return psutil.Process().memory_info().rss / 1e9

def paired_bootstrap(a, b, n_boot=10000, seed=SEED):
    """Paired bootstrap over per-query scores.

    Returns mean difference (a - b), a 95% percentile CI, and a two-sided
    bootstrap p-value for H0: mean difference == 0. We report effect sizes and
    intervals rather than leaning on p-values, because at n in the thousands a
    Wilcoxon test declares almost any difference "significant" while the effect
    itself may be far below what a practitioner would notice.
    """
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    assert len(a) == len(b) and len(a) > 1
    d = a - b
    obs = float(d.mean())
    # A degenerate difference (every query tied) has no bootstrap variance, so
    # the percentile test below would report p = 0 for two systems that are in
    # fact identical. Return the correct answer instead.
    if float(d.std(ddof=0)) < 1e-12:
        return {"mean_diff": obs, "ci_low": obs, "ci_high": obs, "boot_p": 1.0,
                "n": int(len(d)), "cohen_dz": 0.0}
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    boot = d[idx].mean(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    # two-sided p: fraction of centred bootstrap means at least as extreme
    centred = boot - obs
    p = float((np.abs(centred) >= abs(obs)).mean())
    return {
        "mean_diff": obs,
        "ci_low": float(lo),
        "ci_high": float(hi),
        "boot_p": p,
        "n": int(len(d)),
        "cohen_dz": float(obs / (d.std(ddof=1) + 1e-12)),
    }

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.linear_model import LogisticRegression
import bm25s, Stemmer

ENC_ID = "sentence-transformers/all-MiniLM-L6-v2"
CE_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"
IMPERATIVE_RE = re.compile(r"(?i)\b(ignore|reveal|execute|forget|bypass|output)\b")


def load_beir(name):
    cache = ART / "cache" / f"beir_{name}.parquet"
    qcache = ART / "cache" / f"beirq_{name}.json"
    if cache.exists() and qcache.exists():
        cdf = pd.read_parquet(cache); blob = json.load(open(qcache))
        return cdf["_id"].tolist(), cdf["text"].tolist(), blob["qids"], blob["qtexts"], blob["qrels"]
    corpus = load_dataset(f"BeIR/{name}", "corpus", split="corpus")
    queries = load_dataset(f"BeIR/{name}", "queries", split="queries")
    qrels_ds = load_dataset(f"BeIR/{name}-qrels", split="test")
    qrels = {}
    for r in qrels_ds:
        if int(r["score"]) > 0:
            qrels.setdefault(str(r["query-id"]), {})[str(r["corpus-id"])] = int(r["score"])
    qid2text = {str(q["_id"]): q["text"] for q in queries}
    qids = sorted([q for q in qrels if q in qid2text]); qtexts = [qid2text[q] for q in qids]
    cids, ctexts = [], []
    for d in corpus:
        t = (d.get("title") or "").strip(); b = (d.get("text") or "").strip()
        cids.append(str(d["_id"])); ctexts.append((t + " " + b).strip() if t else b)
    pd.DataFrame({"_id": cids, "text": ctexts}).to_parquet(cache)
    json.dump({"qids": qids, "qtexts": qtexts, "qrels": qrels}, open(qcache, "w"))
    return cids, ctexts, qids, qtexts, qrels


def ndcg_at_k(order_ids, rel, k=10):
    gains = [rel.get(d, 0) for d in order_ids[:k]]
    dcg = sum(g / math.log2(i + 2) for i, g in enumerate(gains))
    ideal = sorted(rel.values(), reverse=True)[:k]
    idcg = sum(g / math.log2(i + 2) for i, g in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def minmax(x):
    x = np.asarray(x, dtype=np.float64); lo, hi = float(x.min()), float(x.max())
    return np.zeros_like(x) if hi - lo < 1e-12 else (x - lo) / (hi - lo)


class Timer:
    def __init__(self): self.t = {}
    def __call__(self, k):
        self.k = k; return self
    def __enter__(self):
        if DEVICE == "cuda": torch.cuda.synchronize()
        self.t0 = time.perf_counter(); return self
    def __exit__(self, *a):
        if DEVICE == "cuda": torch.cuda.synchronize()
        self.t[self.k] = self.t.get(self.k, 0.0) + (time.perf_counter() - self.t0) * 1000

## Benchmark loop

Indices are built once per corpus and kept resident, exactly as a deployed system
would — the CIKM candidate-pool code rebuilt a FAISS index and a BM25 index
*inside the per-query loop*, which is why its own timing artifacts vary by an
order of magnitude depending on which script produced them. Timings below exclude
index construction, which is reported separately.

In [ ]:
encoder = SentenceTransformer(ENC_ID, device=DEVICE); encoder.max_seq_length = 256
cross_encoder = CrossEncoder(CE_ID, max_length=320, device=DEVICE)

# A minimal three-feature filter so the filtering stage is timed as configured.
_f_clean = ["placeholder"] * 6
_filter_clf = LogisticRegression().fit(np.array([[0.9, 0, 1.0], [0.2, 1, 0.2]] * 3),
                                       [0, 1] * 3)

rows, build_rows = [], []

for ds in BENCH_DATASETS:
    cids, ctexts, qids, qtexts, qrels = load_beir(ds)
    rng = np.random.default_rng(SEED)
    pick = rng.choice(len(qids), size=min(N_BENCH_QUERIES, len(qids)), replace=False)
    bq = [qtexts[i] for i in sorted(pick)]; bqid = [qids[i] for i in sorted(pick)]

    m0 = rss_gb()
    t0 = time.perf_counter()
    cpath = ART / "cache" / f"emb_{ds}_minilm.npy"
    if cpath.exists():
        emb = np.load(cpath); enc_time = float("nan")
    else:
        emb = encoder.encode(ctexts, batch_size=256, convert_to_numpy=True,
                             normalize_embeddings=True, show_progress_bar=True).astype(np.float16)
        np.save(cpath, emb); enc_time = time.perf_counter() - t0
    mat = torch.from_numpy(emb).to(DEVICE)
    t1 = time.perf_counter()
    stem = Stemmer.Stemmer("english")
    bm = bm25s.BM25(k1=0.9, b=0.4)
    bm.index(bm25s.tokenize(ctexts, stopwords="en", stemmer=stem, show_progress=False),
             show_progress=False)
    bm25_build = time.perf_counter() - t1
    mu = torch.from_numpy(emb[:2000].astype(np.float32)).mean(0)
    mu = (mu / mu.norm()).to(DEVICE)
    clean_mean_len = float(np.mean([len(t) for t in ctexts]))

    build_rows.append({"dataset": ds, "n_docs": len(ctexts),
                       "corpus_encode_s": enc_time, "bm25_build_s": bm25_build,
                       "embedding_mb": emb.nbytes / 1e6,
                       "rss_after_build_gb": rss_gb(), "rss_delta_gb": rss_gb() - m0})

    for depth in DEPTHS:
        for budget in RERANK_BUDGETS:
            if budget > depth:
                continue
            per_q, ndcgs, ce_calls = [], [], []
            for i, q in enumerate(tqdm(bq, desc=f"{ds} d={depth} b={budget}", leave=False)):
                T = Timer()
                with T("encode_query"):
                    qe = encoder.encode([q], convert_to_numpy=True,
                                        normalize_embeddings=True).astype(np.float16)
                with T("dense_search"):
                    qt = torch.from_numpy(qe).to(DEVICE)
                    sc, ix = torch.topk((qt @ mat.T).float(), min(depth, mat.shape[0]), dim=1)
                    d_sc, d_ix = sc[0].cpu().numpy(), ix[0].cpu().numpy()
                with T("bm25_search"):
                    tk = bm25s.tokenize([q], stopwords="en", stemmer=stem, show_progress=False)
                    b_ix, b_sc = bm.retrieve(tk, k=min(depth, len(ctexts)), show_progress=False)
                    b_ix, b_sc = b_ix[0], b_sc[0]
                with T("fuse"):
                    nd, nb = minmax(d_sc), minmax(b_sc)
                    dmap = {int(a): float(b) for a, b in zip(d_ix, nd)}
                    bmap = {int(a): float(b) for a, b in zip(b_ix, nb)}
                    cand = np.array(sorted(set(dmap) | set(bmap)))
                    fs = np.array([0.6 * dmap.get(int(c), 0.) + 0.4 * bmap.get(int(c), 0.)
                                   for c in cand])
                    order = cand[np.argsort(-fs, kind="stable")]
                with T("rerank"):
                    n_ce = 0
                    if budget > 0:
                        head = order[:budget]; n_ce = len(head)
                        cs = cross_encoder.predict([[q, ctexts[j]] for j in head],
                                                   batch_size=128, show_progress_bar=False)
                        order = np.concatenate([head[np.argsort(-cs, kind="stable")],
                                                order[budget:]])
                with T("filter"):
                    top = order[:10]
                    te = encoder.encode([ctexts[j] for j in top], convert_to_numpy=True,
                                        normalize_embeddings=True)
                    feats = np.stack([
                        te @ mu.cpu().numpy(),
                        np.array([1.0 if IMPERATIVE_RE.search(ctexts[j]) else 0.0 for j in top]),
                        np.array([len(ctexts[j]) / clean_mean_len for j in top])], axis=1)
                    _ = _filter_clf.predict_proba(feats)[:, 1]
                if i >= WARMUP:
                    per_q.append(dict(T.t, total=sum(T.t.values())))
                    ce_calls.append(n_ce)
                    ndcgs.append(ndcg_at_k([cids[j] for j in order[:10]], qrels[bqid[i]], 10))
            d = pd.DataFrame(per_q)
            rows.append({
                "dataset": ds, "n_docs": len(ctexts), "depth": depth, "rerank_budget": budget,
                "ndcg10": float(np.mean(ndcgs)),
                "ce_forward_passes_per_query": float(np.mean(ce_calls)),
                "p50_ms": float(d.total.quantile(.50)), "p95_ms": float(d.total.quantile(.95)),
                "p99_ms": float(d.total.quantile(.99)), "mean_ms": float(d.total.mean()),
                **{f"stage_{c}_ms": float(d[c].mean()) for c in d.columns if c != "total"},
            })
    del mat; gc.collect(); torch.cuda.empty_cache()

cost = pd.DataFrame(rows).round(3)
save_csv(cost, "nb5_cost_quality.csv")
save_csv(pd.DataFrame(build_rows).round(3), "nb5_index_build.csv")
save_json(CONFIG, "nb5_config.json")
print(cost[["dataset", "n_docs", "depth", "rerank_budget", "ndcg10",
            "ce_forward_passes_per_query", "p50_ms", "p95_ms", "p99_ms"]].to_string(index=False))

In [ ]:
print("\n=== Stage breakdown (mean ms/query, depth=1000, rerank=100) ===")
sub = cost[(cost.depth == 1000) & (cost.rerank_budget == 100)]
stage_cols = [c for c in cost.columns if c.startswith("stage_")]
print(sub.set_index("dataset")[stage_cols].round(2).to_string())

print("\n=== What the reranker costs per nDCG point ===")
for ds, g in cost[cost.depth == 1000].groupby("dataset"):
    base = g[g.rerank_budget == 0]
    if base.empty:
        continue
    b_ndcg, b_ms = float(base.ndcg10.iloc[0]), float(base.mean_ms.iloc[0])
    for _, r in g[g.rerank_budget > 0].iterrows():
        dn, dm = r.ndcg10 - b_ndcg, r.mean_ms - b_ms
        print(f"  {ds:9s} top-{int(r.rerank_budget):3d}: "
              f"{dn:+.4f} nDCG for {dm:+7.1f} ms  "
              f"({dm/max(dn,1e-9):8.0f} ms per nDCG point)")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6.4, 4.2))
for ds, g in cost[cost.depth == 1000].groupby("dataset"):
    g = g.sort_values("mean_ms")
    ax.plot(g.mean_ms, g.ndcg10, marker="o", label=f"{ds} ({int(g.n_docs.iloc[0]):,} docs)")
    for _, r in g.iterrows():
        ax.annotate(f"CE@{int(r.rerank_budget)}", (r.mean_ms, r.ndcg10),
                    fontsize=7, xytext=(4, -8), textcoords="offset points")
ax.set_xlabel("mean latency per query (ms, persistent index)")
ax.set_ylabel("nDCG@10"); ax.set_xscale("log")
ax.set_title("Cost/quality frontier of cross-encoder reranking")
ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(ART / "results" / f"fig_cost_quality.{ext}", dpi=180, bbox_inches="tight")
print("\nsaved fig_cost_quality.{pdf,png}")
plt.close()

## Reading this

Report `ce_forward_passes_per_query` next to every latency figure. Wall-clock on
a T4 tells a reader nothing about their own hardware; forward passes tell them
everything, and the two together let anyone re-derive the first from the second.

The "ms per nDCG point" line is the honest version of the CIKM latency claim. If
reranking the top-100 costs 200 ms for +0.05 nDCG on a 58K corpus, that is a
defensible engineering trade and should be stated as such — not as "sub-300 ms,
therefore practical", which was a claim about a synthetic 12-word vocabulary.